# Using Z3 as a Theorem Prover

In principle, it should be possible to use Z3 as a theorem prover.
In this example, we check whether Z3 is able to prove that the three 
axioms of a group, which are
* $\mathrm{e} \cdot x = x$
* $x^{-1} \cdot x = \mathrm{e}$
* $(x \cdot y) \cdot z = x \cdot (y \cdot z)$

imply that the theorem 

 * $(x \cdot y)^{-1} = y^{-1} \cdot x^{-1}$

holds.

In [1]:
// load the formatting setup and the style sheet, install Z3 if necessary
#load "../setup.fsx"
installZ3 ()
HTML(System.IO.File.ReadAllText "../style.css")

The below script needs to be able to find the current output cell; this is an easy method to get it.

In [2]:
// load Z3
#load "../z3.fsx"
open Microsoft.Z3

First, we declare an abstract type with the name `Group`.  The function `DeclareSort` is defined in `z3.fsx`.

In [3]:
let Group = DeclareSort "Group"

Next, we declare the neutral element `e` and three variables `x`, `y`, and `z`.
All of these are elements of the type `Group`.

In [4]:
let e = Const "e" Group
let x = Const "x" Group
let y = Const "y" Group
let z = Const "z" Group

Next, we declare the function `mul` that represents the multiplication of two group elements, i.e. `mul x y` is interpreted as $x \cdot y$
and the function `inv` that represents the inverse of a group element, i.e. `inv x` is interpreted as $x^{-1}$.  The Z3 functions are declared with `Function`, whose arguments are the name, the list of argument sorts, and the result sort.  The F# functions `mul` and `inv` apply these Z3 functions to their arguments.

In [5]:
let mulF = Function "mul" [ Group; Group ] Group
let invF = Function "inv" [ Group ] Group
let mul (a: Expr) (b: Expr) : Expr = mulF.Apply(a, b)
let inv (a: Expr) : Expr = invF.Apply(a)

We create a solver.  The time that `Z3` needs for this problem varies greatly:  Sometimes the problem is solved in about a minute, sometimes `Z3` does not find a proof after half an hour.  Therefore, we set a *timeout* of five minutes.  If the timeout is reached, `Check` returns `unknown`.  The timeout is given in milliseconds as an unsigned integer, which is written with the suffix `u`.

In [6]:
let S = newSolver ()
let parameters = ctx.MkParams()
parameters.Add("timeout", 300000u) |> ignore
S.Parameters <- parameters

We add the axioms to the solver.  

In [7]:
S.Add(ForAll [ x ] (mul e x .= x))
S.Add(ForAll [ x ] (mul (inv x) x .= e))
S.Add(ForAll [ x; y; z ] (mul x (mul y z) .= mul (mul x y) z))

Finally, we add the negation of the theorem that we want to prove.  

In the Python version of this notebook, the negation of the commutative law is added as well.  Logically, this does not change the result:  As the commutative law does not follow from the axioms, the negation of the commutative law is consistent with the axioms.  However, it makes the problem much harder for `Z3`:  With this additional formula, `Z3` (version 5.1) gives up after about eight minutes and returns `unknown`.  Therefore, the next cell is not executed.  You can execute it to see this effect.

In [ ]:
S.Add(Not (ForAll [ x; y ] (mul x y .= mul y x)))

Now we add the negation of the theorem.

In [8]:
S.Add(Not (ForAll [ x; y ] (inv (mul x y) .= mul (inv y) (inv x))))

The property `Assertions` returns the formulas that have been added to the solver.

In [9]:
for a in S.Assertions do
    print a

∀x: mul(e, x) = x

∀x: mul(inv(x), x) = e

∀x, y, z: mul(x, mul(y, z)) = mul(mul(x, y), z)

¬(∀x, y: inv(mul(x, y)) = mul(inv(y), inv(x)))

If the theorem is true, then the constraints given to the solver `S` should be unsatisfiable.

**Note:** With `Z3` version 5.1, the running time of the next cell is unpredictable.  In our tests, it either took about one minute or `Z3` did not terminate within half an hour, where the timeout was not always respected.  Therefore, the next two cells have not been executed when this notebook was created.  The notebook `Z3-Induction.ipynb` contains the same proof, which `Z3` found in about one minute there.

In [ ]:
#!time
let result = S.Check()

In [ ]:
match result with
| Status.UNSATISFIABLE -> printfn "The theorem is valid!"
| Status.SATISFIABLE   -> printfn "The theorem is not valid!"
| _                    -> printfn "Z3 is not able to decide whether this theorem is true."